In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import GDLModules.util as util
import plotly.express as px
import torch.nn as nn
import torch
import GDLModules.A_CAE as A_CAE
import GDLModules.A_AE as A_AE
from datasets import load_dataset
import matplotlib.pyplot as plt
from sklearn.datasets import make_s_curve

import copy
import panel as pn

pn.extension("plotly", comms="ipywidgets")

d:\Miniconda3\envs\test\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_matrix, issparse


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
s_data, _ = make_s_curve(n_samples=1000, noise=0.05, random_state=0)
s_data_test, _ = make_s_curve(n_samples=1000, noise=0.05, random_state=1)
util.plot3d([(s_data, "S-Curve Dataset")], title="Double Torus") 

In [5]:
torus_data = util.generate_torus(10000)
torus_data_test = util.generate_torus(10000)
util.plot3d([(torus_data, "torus data")], title="Double Torus")

In [6]:
X_train = torch.tensor(torus_data, dtype=torch.float32).to(device)
X_test = torch.tensor(torus_data_test, dtype=torch.float32).to(device)
X_loader_train = torch.utils.data.DataLoader(X_train, batch_size=1024, shuffle=True)
X_loader_test = torch.utils.data.DataLoader(X_test, batch_size=1024, shuffle=True)

In [7]:

def reconstruct(model_name, data):
    if model_name == "ae":
        ae.to(device)
        reconstructed = ae.decode(ae.encode(data)).cpu().detach().numpy()
        reconstructed.shape
        return reconstructed
    if model_name == "cae":
        cae.to(device)
        probabilities = cae.true_predict(data)
        chart_indexes = torch.argmax(probabilities, dim=1).to(device)
        reconstructed = cae.decode(cae.encode(data, chart_indexes), chart_indexes).cpu().detach().numpy()

        return reconstructed, chart_indexes


In [8]:
ae = A_AE.AE(3,2,15)
A_AE.train_wae(num_epochs=5, x_train_loader=X_loader_train, network=ae, device=device, mmd_weight=1.0)

100%|██████████| 5/5 [00:00<00:00,  7.27it/s]


In [ ]:
ae.to(device)
encoded = ae.encode(X_test).cpu().detach().numpy()
encoded.shape
plt.scatter(encoded[:, 0], encoded[:, 1], c='blue', alpha=0.5)

In [ ]:
reconstructed = reconstruct("ae", X_train[0:10000])
util.plot3d([(reconstructed, "reconstructed")], title="Double Torus")

In [9]:
cae = A_CAE.CAE(x_dim=3, z_dim=2, hidden_structure=[15,15], chart_amount=10, activation=nn.Tanh()).to(device)

A_CAE.pre_train_cae(num_epochs=50, x_train=X_train, network=cae, device = device, lr=1e-2)


100%|██████████| 50/50 [00:01<00:00, 35.61it/s]


In [11]:
pretrained = copy.deepcopy(cae)

In [ ]:
cae = copy.deepcopy(pretrained)
diag = A_CAE.train_cae(num_epochs=200, x_train_loader=X_loader_train, network=cae, device=device, lr=1e-2, best_k=1, mmd_weight=0.0, lipschitz_weight=0.0)
posttrained = copy.deepcopy(cae)

In [ ]:
cae = posttrained
reconstructed, alpha = reconstruct("cae", X_train[0:10000])
util.plot3d([(reconstructed, "reconstructed")], title="Double Torus", color=[alpha.detach().numpy()])

In [ ]:
cae = copy.deepcopy(posttrained)
diag = A_CAE.train_cae(num_epochs=200, x_train_loader=X_loader_train, network=cae, device=device, lr=1e-3, best_k=1, mmd_weight=0.0, lipschitz_weight=0.05)
lipschitz_trained = copy.deepcopy(cae)

In [ ]:


from GDLModules.viser_viz import open_split_view


sw = open_split_view(X_test, cae)
print("Press Enter to close the viewer...")
input()
sw.close()


In [ ]:
import panel as pn
import plotly.graph_objects as go
import threading
import time

from IPython.display import display
cae = copy.deepcopy(pretrained)



stop_event = threading.Event()
thread = threading.Thread(target=A_CAE.train_cae, args=(200, X_loader_train, cae, device), kwargs={"lr": 1e-2, "best_k": 1, "mmd_weight": 0.0, "lipschitz_weight": 0.0, "stop_event": stop_event})
thread.start()




# Keep the same evaluation points throughout training.
plot_data = X_test[:1000].detach()
plot_data = plot_data.to(next(cae.parameters()).device)

@torch.no_grad()
def plot_values():
    # Run each chart on the whole sample, avoiding per-point network calls.
    reconstructions = torch.stack([
        cae.decode(cae.encode(plot_data, i), i)
        for i in range(cae.chart_amount)
    ], dim=1)  # (N, charts, 3)

    errors = (
        (reconstructions - plot_data[:, None, :]) ** 2
    ).sum(dim=-1)

    charts = errors.argmin(dim=1)
    rows = torch.arange(len(plot_data), device=plot_data.device)
    reconstructed = reconstructions[rows, charts]

    return reconstructed.cpu().numpy(), charts.cpu().numpy()

points, colors = plot_values()
fig = util.plot3d(
    [(points, "reconstructed")],
    title="Training reconstruction",
    color=[colors],
)
fig.update_layout(uirevision="keep-camera")

fig.update_scenes(
    aspectmode="cube",
    xaxis_range=[-1.5, 1.5],
    yaxis_range=[-1.5, 1.5],
    zaxis_range=[-1.5, 1.5],
)

plot = pn.pane.Plotly(fig, height=600)
display(plot)

def update():
    # Capture the browser's camera before changing the figure.
    camera = (plot.relayout_data or {}).get("scene.camera")
    points, colors = plot_values()

    with fig.batch_update():
        if camera is not None:
            fig.layout.scene.camera = camera

        fig.data[0].x = points[:, 0]
        fig.data[0].y = points[:, 1]
        fig.data[0].z = points[:, 2]
        fig.data[0].marker.color = colors

callback = pn.state.add_periodic_callback(update, period=2000)


BokehModel(combine_events=True, render_bundle={'docs_json': {'0391ed47-2a96-4ec5-b8f7-a433f0aa7e5e': {'version…

  6%|▌         | 12/200 [00:20<05:17,  1.69s/it, loss=0.6820%, lr=0.010000, rec_loss=0.6623%]


Training stopped.


In [20]:
stop_event.set()  # Signal the training thread to stop
callback.stop()

In [ ]:


pn.extension("plotly", comms="ipywidgets")

# Shared data
data = []
stop = threading.Event()


# Something running independently
def worker():
    for i in range(1000):
        if stop.is_set():
            break
        data.append(i ** 0.5)
        time.sleep(0.1)
        

thread =threading.Thread(target=worker, daemon=True)
thread.start()

# Plot
fig = go.Figure()
fig.add_scatter(x=[], y=[])

plot = pn.pane.Plotly(fig)

# Periodically refresh the plot
def update():
    fig.data[0].x = list(range(len(data)))
    fig.data[0].y = data.copy()
    plot.param.trigger("object")

callback = pn.state.add_periodic_callback(update, period=200)

plot